In [13]:
from pathlib import Path 
import json 
from collections import defaultdict
#cur_dir = Path.absolute()

with open("annos.json", "r") as f:
    data = json.load(f)

keys = data.keys()
#print(keys) > dict_keys(['info', 'licenses', 'categories', 'images', 'annotations'])

images = data['images']
annotations = data['annotations']

total_images = len(images)
total_annotations = len(annotations)

#print(total_images)
#print(total_annotations)

images_by_id = images.copy()

img_w_h_distri = defaultdict(int)


for image in images_by_id:
    image.pop('license')
    image.pop('date_captured')
    image.pop('file_name')
    image.pop('extra')
    #img_id = image['id']
    w = image['width']
    h = image['height']
    w_h = (w, h)
    img_w_h_distri[w_h] += 1

annos_by_id = defaultdict(list)
for anno in annotations:
    img_id = int(anno['image_id'])
    annos_by_id[img_id].append(anno)


# Verify the len of all the annos in the dict add up to the total anno
anno_counter = 0
anno_per_image = defaultdict(int)
for k in annos_by_id.keys():
    anno_len = len(annos_by_id[k])
    anno_counter += anno_len
    anno_per_image[anno_len] += 1


# count images with 0/1/2/+ plates 

# count width/height distribution 
img_w_h_distri = dict(sorted(img_w_h_distri.items(), key = lambda item: item[1], reverse = True))
anno_per_image

defaultdict(int, {1: 95651, 2: 2223, 6: 41, 3: 500, 4: 214, 5: 29})

In [3]:
images_by_id[0]

{'id': 0, 'height': 303, 'width': 472}

In [14]:
annos_by_id[0]

[{'id': 1,
  'image_id': 0,
  'category_id': 1,
  'bbox': [200, 176, 53, 19.785],
  'area': 1048.615,
  'segmentation': [],
  'iscrowd': 0}]

In [51]:
for k,v in annos_by_id.items():
    if len(v) > 1:
        pass
# 328 340 343 391 406 409 412 429 433 465 495

In [30]:
# add annos_by_id into images_by_id
for image in images_by_id: 
    i_d = image['id']
    image['annotation'] = annos_by_id.get(i_d)

In [29]:
images_by_id[328]

{'id': 328,
 'height': 720,
 'width': 960,
 'annotation': [{'id': 329,
   'image_id': 328,
   'category_id': 1,
   'bbox': [85, 534, 150.467, 100.104],
   'area': 15062.271,
   'segmentation': [],
   'iscrowd': 0},
  {'id': 330,
   'image_id': 328,
   'category_id': 1,
   'bbox': [887, 502, 39.696, 47.289],
   'area': 1877.194,
   'segmentation': [],
   'iscrowd': 0}]}

In [55]:
annos_by_id.get(333)

[{'id': 335,
  'image_id': 334,
  'category_id': 1,
  'bbox': [188, 496, 108.304, 41],
  'area': 4440.452,
  'segmentation': [],
  'iscrowd': 0}]

In [67]:
# calculate bbox relative area to the image area 
# need image area and anno area 

# there are more than 1 anno in some images
# what do you do for that case? 

for image in images_by_id:
    i_w = image['width']
    i_h = image['height']
    img_area = i_w * i_h
    # select the anno key 
    anno = image['annotation']
    bbox_data = defaultdict(dict)
    
    try: 
        for i, data in enumerate(anno):
            b_w = data['bbox'][2]
            b_h = data['bbox'][3]
            area = data['area']
            d = {}
            d['w'] = b_w 
            d['h'] = b_h 
            d['a'] = area 

            bbox_data[i] = d 

    except Exception as e:
        # images without annotations 
        pass 

    # calcualte the area 
    #relative_area = [(bb_area / img_area) * 100 for bb_area in temp_bbox_areas]






In [77]:
bbox_data = []

for image in images_by_id:

    # Get image dimensions
    i_d = image['id']
    i_w = image['width']
    i_h = image['height']

    # Calculate image area
    img_area = i_w * i_h

    # Get annotations
    annos = image.get('annotation', [])
    
    # Loop through every annotation in this image
    try: 
        for anno in annos:
    
            # COCO bbox format = [x, y, width, height]
            b_w = anno['bbox'][2]
            b_h = anno['bbox'][3]
    
            # Calculate bbox area
            #bbox_area = b_w * b_h
    
            bbox_area = anno['area']
    
            # Calculate relative area
            relative_area = bbox_area / img_area
    
            # Convert to percentage
            relative_area_percent = round(relative_area * 100, 4)
    
            # Store the result
            bbox_data.append({
                'image_id': i_d,
                'bbox_width': b_w,
                'bbox_height': b_h,
                'bbox_area': bbox_area,
                'image_area': img_area,
                'relative_area': relative_area,
                'relative_area_percent': relative_area_percent
            })
    except Exception as e:
        pass



bbox_data 

[{'image_id': 0,
  'bbox_width': 53,
  'bbox_height': 19.785,
  'bbox_area': 1048.605,
  'image_area': 143016,
  'relative_area': 0.007332081725121665,
  'relative_area_percent': 0.73},
 {'image_id': 1,
  'bbox_width': 47,
  'bbox_height': 12,
  'bbox_area': 564,
  'image_area': 140400,
  'relative_area': 0.004017094017094017,
  'relative_area_percent': 0.4},
 {'image_id': 2,
  'bbox_width': 109.681,
  'bbox_height': 92.807,
  'bbox_area': 10179.164567,
  'image_area': 127556,
  'relative_area': 0.07980153475336323,
  'relative_area_percent': 7.98},
 {'image_id': 3,
  'bbox_width': 97.393,
  'bbox_height': 94.711,
  'bbox_area': 9224.188423,
  'image_area': 124080,
  'relative_area': 0.07434065460186975,
  'relative_area_percent': 7.43},
 {'image_id': 4,
  'bbox_width': 94.348,
  'bbox_height': 91,
  'bbox_area': 8585.668,
  'image_area': 135240,
  'relative_area': 0.06348467908902691,
  'relative_area_percent': 6.35},
 {'image_id': 5,
  'bbox_width': 47.237,
  'bbox_height': 16.348,
 

# 